# CrisperWhisper: word-for-word transcripts with fillers (C4 study)

Turns each student's Zoom recording into word timings that keep "um" and "uh"
(marked `[UM]` and `[UH]`). The analysis tool reads these files.

**Before you start:** Runtime menu, Change runtime type, choose **T4 GPU**, Save.

**Steps:** run each cell in order (or Runtime, Run all). When asked, upload only the
`*_student.m4a` files. At the end `words.zip` downloads. Unzip it into your recordings
folder, next to the `.m4a` files, so each `P03_student.m4a` has a `P03_student.words.json`.

CrisperWhisper weights are free for non-commercial research use. Uploading sends the
audio to Google, which your consent form must mention. Delete the files from Colab when
done (Runtime, Disconnect and delete runtime).

In [ ]:
# 1. Install (about 2 minutes)
!pip -q install "crisperwhisper[ct2]"
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# 2. Upload the *_student.m4a files
from google.colab import files

uploaded = files.upload()
print(sorted(uploaded))

In [ ]:
# 3. Transcribe (a 10-minute recording takes a few minutes on the T4 GPU)
import json
import subprocess
import zipfile
from pathlib import Path

from crisperwhisper import CrisperWhisperModel

model = CrisperWhisperModel()  # large model, verbatim mode
outputs = []
for name in sorted(uploaded):
    path = Path(name)
    if path.suffix.lower() not in {".m4a", ".mp3", ".mp4", ".wav", ".webm", ".ogg", ".flac"}:
        continue
    wav = path.with_suffix(".16k.wav")  # Zoom's m4a is converted to 16 kHz mono WAV first
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(path),
            "-ac",
            "1",
            "-ar",
            "16000",
            str(wav),
        ],
        check=True,
    )
    result = model.transcribe(str(wav), language="en", word_timestamps=True)
    words = [
        {"start": float(w.start), "end": float(w.end), "word": str(w.word)} for w in result.words
    ]
    out = Path(path.stem + ".words.json")
    out.write_text(
        json.dumps(
            {"source": "CrisperWhisper", "file": name, "text": result.text, "words": words},
            indent=1,
        )
    )
    fillers = sum(
        1
        for w in words
        if w["word"].strip("[] .,").lower() in {"uh", "um", "uhm", "er", "erm", "hmm"}
    )
    print(f"{out}: {len(words)} words, {fillers} fillers")
    outputs.append(out)

In [ ]:
# 4. Download the results
with zipfile.ZipFile("words.zip", "w") as z:
    for out in outputs:
        z.write(out)
files.download("words.zip")

If step 1 fails, replace `crisperwhisper[ct2]` with `crisperwhisper[transformers]` and run
it again. If Colab says no GPU is available, try later, or run the analysis tool with
`--transcriber local` on your laptop instead.